# Scenario 

<div style='color: #3366ff'>You're trying to download tweets and process them for sentiments: </div>
    - "I have a @Microsoft Surface Pro 4 and LOVE it!"
    - "#surfacepro @surface Why is this so expensive?"

<div style='color: #3366ff'>How do we solve this problem? </div>

* Strategy 1: Download all tweets and then process them 
    - `download, loop [ process ]`
* Strategy 2: Download each tweet in small batches 
    - `loop [ download tweet, process ] 

Need a function that can implement strategy 1 but does not need to keep everything in memory! 

Same design pattern is used in many cases. For instance, machine learning: load images and train machine based on them. 
* There are thousands (or even millions) of images so you can't load them all in memory. 
* The processes are complicated so you don't want to couple loading logic with learing. 

# Generators

Let's first see waht we are trying to do and then define the generators

In [3]:
for i in [0, 1, 2, 3, 4]: # keeps the list in memeory
    print(i)              # process each item

0
1
2
3
4


In [4]:
for i in range(5): # does not generate all five elements in memory!
    print(i)

0
1
2
3
4


In [7]:
type(range(5)) # not a list but a generator object

range

In [8]:
def myRange(n):
    x = 0
    while x < n:
        yield x # 'yield' turns a fucntion into a generator
        x += 1

In [9]:
type(myRange(5))

generator

In [10]:
for i in myRange(5):
    print(i)

0
1
2
3
4


In [11]:
def countdown(n):
    while n > 0:
        print("Computing next number...")
        yield n
        n -= 1

In [12]:
for i in countdown(5):
    print(i)

Computing next number...
5
Computing next number...
4
Computing next number...
3
Computing next number...
2
Computing next number...
1


In [21]:
v = countdown(5)

In [22]:
next(v)

Computing next number...


5

In [24]:
import random

def random_gen(low, high, num):
    i = 0
    while i < num:
        yield random.randrange(low, high)
        i += 1

In [25]:
r = random_gen(0,100, 5)

In [26]:
type(r)

generator

In [27]:
list(r)

[93, 21, 52, 69, 13]

In [28]:
def random_gen_inf(low, high):
    while True:
        yield random.randrange(low, high)

In [29]:
r = random_gen_inf(0, 100)

In [34]:
next(r)

38

In [35]:
%time v = [ i**2 for i in range(10000000) ]

CPU times: user 522 ms, sys: 167 ms, total: 688 ms
Wall time: 687 ms


In [36]:
print(v[:10])

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


In [46]:
%time g = ( i**2 for i in range(10000000) ) # this is not a tuple but generator

CPU times: user 7 μs, sys: 1 μs, total: 8 μs
Wall time: 10.5 μs


In [41]:
g

<generator object <genexpr> at 0x7ba3501d7c60>

In [45]:
next(g)

9

# Real world example

In [49]:
wwwlog = open("access-log")
for line in wwwlog:
    print(line.rsplit(None, 1))
    break

['140.180.132.213 - - [24/Feb/2008:00:08:59 -0600] "GET /ply/ply.html HTTP/1.1" 200', '97238']


In [50]:
wwwlog = open('access-log')
total = 0

In [51]:
for line in wwwlog:
    bytestr = line.rsplit(None, 1)[1]
    if bytestr != '-':
        total += int(bytestr)

print("Total", total)

Total 230741830


The generator way

In [1]:
wwwlog     = open('access-log')
bytecolumn = (line.rsplit(None, 1)[1] for line in wwwlog)
bytes      = (int(x) for x in bytecolumn if x != '-')

print("Total", sum(bytes))

Total 230741830


# Tailing a File

it's similar to `tail` command in linux

In [59]:
import time
def follow(thefile):
    thefile.seek(0, 2)                # Go to the end of the file
    while True:
        line = thefile.readline()
        if not line:
            time.sleep(0.1)           # sleep briefly
            continue
        yield line

In [60]:
logfile = open('test-log')

In [61]:
loglines = follow(logfile)

In [62]:
type(loglines)

generator

In [63]:
for line in loglines:
    print(line, )

    if line[:1] == '.':
        break

something


.something
